# 03 — Modeling

Step-by-step walkthrough of the experiment pipeline:
CV splits, per-fold training, evaluation, and SHAP interpretability.

In [1]:
# Automatic module reloading
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.modeling.config import resolve_experiment
from src.modeling.dataset import build_modeling_dataset
from src.modeling.features import (
    build_feature_pipeline,
    get_feature_names,
    build_feature_group_index,
    warn_loso_onehot,
)
from src.modeling.models import build_model
from src.modeling.cv import make_splits, get_fold_group_label
from src.modeling.evaluate import (
    evaluate_fold,
    aggregate_metrics,
    per_group_metrics,
)
from src.modeling.interpret import compute_shap_values, plot_shap_summary

sns.set_theme(style="whitegrid")

## 1. Load experiment config

In [2]:
exp_cfg, ds_cfg, mdl_cfg = resolve_experiment(
    ROOT / "config/experiments/aero_xgboost.yaml", root=ROOT
)
print(f"Task: {exp_cfg.task}")
print(f"Model: {mdl_cfg.type}")
print(f"CV: {exp_cfg.cv.strategy}")

Task: growth_class
Model: xgboost
CV: ['loso', 'lomo']


## 2. Load modeling dataset

In [3]:
interim_path = ROOT / "data" / "interim" / ds_cfg.output_name
df = pd.read_csv(interim_path)
print(f"Shape: {df.shape}")
df.head()

Shape: (672, 17)


,exp_id,date,well_id,row,col,well_index,strain_name,media_name,atmosphere,temperature,replicate,max_od,auc,growth_rate,lag_time_h,time_to_max_od_h,growth_class
0,aerobic_a_25,20260325,a_1_0,a,1,0,escherichia coli k12,lb,aerobic,25.0,1,1.374833,81.339258,0.813405,0.235192,30.666667,1
1,aerobic_a_25,20260325,a_2_1,a,2,1,escherichia coli k12,mb,aerobic,25.0,1,0.435893,22.599028,0.610674,0.448222,14.333333,1
2,aerobic_a_25,20260325,a_3_2,a,3,2,escherichia coli k12,nb,aerobic,25.0,1,0.683583,39.182484,0.546716,0.166613,12.333333,1
3,aerobic_a_25,20260325,a_4_3,a,4,3,escherichia coli k12,bhi,aerobic,25.0,1,1.569167,91.604861,0.797525,0.390312,44.000000,1
4,aerobic_a_25,20260325,a_5_4,a,5,4,escherichia coli k12,mrs,aerobic,25.0,1,1.516048,52.588345,0.425336,23.873531,67.000000,1


## 3. Single CV fold walkthrough

In [4]:
task = exp_cfg.task
strategy = exp_cfg.cv.strategy[0]  # first strategy

warn_loso_onehot(ds_cfg, strategy)

# Get first fold
splits = list(make_splits(strategy, df, y=df[task].values))
train_idx, test_idx = splits[0]
fold_label = get_fold_group_label(strategy, df, test_idx)
print(f"Strategy: {strategy}")
print(f"Fold 1: held-out = {fold_label}")
print(f"Train: {len(train_idx)}, Test: {len(test_idx)}")

2026-06-24 13:00:58.702 | WARNING  | src.modeling.features:warn_loso_onehot:157 - LOSO CV with only one-hot strain_name: the held-out strain will have an all-zero feature vector. Add genome/taxonomy features for meaningful generalization.
2026-06-24 13:00:59.055 | INFO     | src.modeling.cv:make_splits:57 - LOSO: 26 folds (one per strain)


Strategy: loso
Fold 1: held-out = acetobacter aceti
Train: 648, Test: 24


In [5]:
# Fit feature pipeline on train fold only
df_train = df.iloc[train_idx]
df_test = df.iloc[test_idx]

ct, _, fgm = build_feature_pipeline(ds_cfg, df_train, root=ROOT)
ct.fit(df_train)
X_train = ct.transform(df_train)
X_test = ct.transform(df_test)
y_train = df_train[task].values
y_test = df_test[task].values

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

X_train: (648, 39), X_test: (24, 39)


In [6]:
# Build and fit model
model = build_model(mdl_cfg, task, seed=exp_cfg.seed)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1] if task == "growth_class" else None

# Evaluate
metrics = evaluate_fold(y_test, y_pred, y_prob, task)
print(f"Fold 1 metrics: {metrics}")

/scratch/chinasse/MicrobeNet/.venv/lib/python3.11/site-packages/xgboost/core.py:751: UserWarning: [13:03:43] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Fold 1 metrics: {'auroc': 0.6526315789473685, 'pr_auc': 0.46904761904761905, 'f1': 0.37037037037037035, 'mcc': 0.1546720562224365, 'balanced_accuracy': 0.5526315789473684}


## 4. Full CV loop

In [7]:
all_metrics = []
all_preds = []

for fold_num, (train_idx, test_idx) in enumerate(splits, 1):
    fold_label = get_fold_group_label(strategy, df, test_idx)
    
    df_tr, df_te = df.iloc[train_idx], df.iloc[test_idx]
    ct, _, fgm = build_feature_pipeline(ds_cfg, df_tr, root=ROOT)
    ct.fit(df_tr)
    X_tr, X_te = ct.transform(df_tr), ct.transform(df_te)
    y_tr, y_te = df_tr[task].values, df_te[task].values
    
    mdl = build_model(mdl_cfg, task, seed=exp_cfg.seed)
    mdl.fit(X_tr, y_tr)
    yp = mdl.predict(X_te)
    yprob = mdl.predict_proba(X_te)[:, 1] if task == "growth_class" else None
    
    m = evaluate_fold(y_te, yp, yprob, task)
    m["fold"] = fold_num
    m["fold_label"] = fold_label
    all_metrics.append(m)
    
    pred_df = pd.DataFrame({"y_true": y_te, "y_pred": yp, "fold_label": fold_label})
    if yprob is not None:
        pred_df["y_prob"] = yprob
    if "strain_name" in df.columns:
        pred_df["strain_name"] = df_te["strain_name"].values
    if "media_name" in df.columns:
        pred_df["media_name"] = df_te["media_name"].values
    all_preds.append(pred_df)

agg = aggregate_metrics(all_metrics)
print(f"\n{strategy} aggregate:")
for k, v in agg.items():
    if isinstance(v, dict):
        print(f"  {k}: {v['mean']:.4f} +/- {v['std']:.4f}")

/scratch/chinasse/MicrobeNet/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/scratch/chinasse/MicrobeNet/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/scratch/chinasse/MicrobeNet/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/scratch/chinasse/MicrobeNet/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/scratch/chinasse/MicrobeNet/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in

TypeError: ufunc 'isnan' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''

## 5. Per-group breakdown

In [ ]:
predictions_df = pd.concat(all_preds, ignore_index=True)

if "strain_name" in predictions_df.columns:
    pg = per_group_metrics(predictions_df, "strain_name", task)
    display(pg)

## 6. SHAP interpretability

In [ ]:
# Refit pipeline and model on all data
ct_full, _, fgm_full = build_feature_pipeline(ds_cfg, df, root=ROOT)
ct_full.fit(df)
X_full = ct_full.transform(df)
y_full = df[task].values
feature_names = get_feature_names(ct_full)

model_full = build_model(mdl_cfg, task, seed=exp_cfg.seed)
model_full.fit(X_full, y_full)

sv = compute_shap_values(model_full, X_full, feature_names)

import shap
shap.plots.beeswarm(sv, max_display=15, show=True)

In [ ]:
# Group importance
from src.modeling.interpret import compute_group_importance

group_idx = build_feature_group_index(feature_names, fgm_full)
gi = compute_group_importance(sv, group_idx)

fig, ax = plt.subplots(figsize=(7, 4))
groups = sorted(gi, key=gi.get, reverse=True)
ax.barh(groups, [gi[g] for g in groups])
ax.set_xlabel("Mean |SHAP value|")
ax.set_title("Feature Group Importance")
ax.invert_yaxis()
plt.tight_layout()
plt.show()